In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from tqdm import tqdm

# Parâmetros fixos
beta = 0.2
gamma = 0.6
rho   = 0.1

# Condições iniciais
L0 = 0.5
Z0 = 0.25
D0 = 0.0

# Tempo de simulação
t_span = (0, 100)
t_eval = np.linspace(*t_span, 1000)

# Malha de parâmetros
alphas = np.linspace(0.0, 0.75, 10)
deltas = np.linspace(0.0, 0.75, 10)

resultados = np.zeros((len(alphas), len(deltas)))

# Função do modelo
def modelo_zumbi(t, y, alpha, beta, gamma, delta, rho):
    L, Z, D = y
    dLdt = (alpha - beta) * L - gamma * L * Z
    dZdt = rho * beta * L + (gamma - delta) * L * Z
    dDdt = (1 - rho) * beta * L + delta * L * Z
    return [dLdt, dZdt, dDdt]

# Loop sobre a malha de parâmetros
for i, alpha in tqdm(enumerate(alphas), total=len(alphas)):
    for j, delta in enumerate(deltas):
        sol = solve_ivp(
                        modelo_zumbi,
                        t_span,
                        [L0, Z0, D0],
                        t_eval=np.linspace(0, 100, 200),  # menos pontos
                        args=(alpha, beta, gamma, delta, rho),
                        method='RK23',                    # mais rápido que RK45
                        rtol=1e-2,
                        atol=1e-4)
        L_final = sol.y[0, -1]
        Z_final = sol.y[1, -1]
        if L_final + Z_final > 0:
            indicador = (L_final - Z_final) / (L_final + Z_final)
        else:
            indicador = -1  # tudo morreu
        resultados[i, j] = indicador

plt.figure(figsize=(10, 6))
plt.imshow(resultados, origin='lower', aspect='auto',
           extent=[deltas[0], deltas[-1], alphas[0], alphas[-1]],
           cmap='seismic', vmin=-1, vmax=1)
plt.colorbar(label='(L - Z) / (L + Z)')
plt.xlabel('δ (Força dos humanos)')
plt.ylabel('α (Taxa de nascimento)')
plt.title('Espaço de parâmetros (Modelo Contínuo)')
plt.grid(False)
plt.show()
